# 🦷 YOLO26 Nano — Green Cloth Surgical Instruments (14 classes)

Detection on **green cloth with shadows** (shadows make bounding hard). Dataset is **COCO Segmentation** from Roboflow (`dataset/train/_annotations.coco.json` + `dataset/valid/_annotations.coco.json`). This notebook converts it to **YOLO format** and trains **YOLO26 nano** (fallback to YOLO11 nano if 26 not yet in ultralytics).

Pipeline: `COCO → YOLO txt + yaml → YOLO26n train → validate → predict`  
All 14 classes use `bbox_margin` crop logic from the classification repo — here YOLO learns the box directly, so no manual length feature is needed.

## 0) Install dependencies

In [ ]:
%pip install -q ultralytics pycocotools opencv-python-headless
import ultralytics; print(ultralytics.__version__)

## 1) Prepare data — pick **one** method and run this cell

- **A) Upload zip** (export from Roboflow then zip `dataset/`)
- **B) Google Drive** (already uploaded)
- **C) Roboflow download snippet** (paste your API code)

In [ ]:
DATA_DIR = "/content/dataset"  # must contain train/_annotations.coco.json and valid/_annotations.coco.json
import os
for sp in ["train","valid"]:
    p = os.path.join(DATA_DIR, sp, "_annotations.coco.json")
    print(sp, "exists" if os.path.exists(p) else "MISSING", p)

# ── A) upload zip ──
# from google.colab import files
# up = files.upload()
# import zipfile
# with zipfile.ZipFile(list(up.keys())[0]) as z:
#     z.extractall("/content")

# ── B) Drive ──
# from google.colab import drive; drive.mount('/content/drive')
# import shutil, pathlib
# shutil.unpack_archive("/content/drive/MyDrive/dataset.zip", "/content")

# ── ls check ──
import pathlib; print(list(pathlib.Path(DATA_DIR).rglob("*.jpg"))[:3])

## 2) Convert COCO → YOLO (detection)

Writes `dataset_yolo/{images,labels}/{train,valid}/` + `dataset_yolo/dataset.yaml`.  
Mapping: `label = index of sorted class names` (same as `dataset.py` in the classification repo, stable across `category_id` order).

In [ ]:
import json, os, shutil, pathlib, cv2
from pathlib import Path

SRC = Path(DATA_DIR)
DST = Path("/content/dataset_yolo")
for sp in ["train","valid"]:
    (DST/"images"/sp).mkdir(parents=True, exist_ok=True)
    (DST/"labels"/sp).mkdir(parents=True, exist_ok=True)

def coco_to_yolo(coco_path: Path, split: str):
    data = json.load(open(coco_path, encoding="utf-8"))
    cats = sorted([c["name"] for c in data["categories"]])
    name_to_id = {n:i for i,n in enumerate(cats)}
    id_to_name = {c["id"]: c["name"] for c in data["categories"]}
    # image_id -> file_name, w, h
    img_info = {im["id"]: im for im in data["images"]}
    # group anns by image
    from collections import defaultdict
    g = defaultdict(list)
    for ann in data["annotations"]:
        g[ann["image_id"]].append(ann)
    for img_id, im in img_info.items():
        w, h = im["width"], im["height"]
        fname = Path(im["file_name"]).name
        src_img = SRC / split / fname
        # fallback: Roboflow sometimes stores file_name with subfolder
        if not src_img.exists():
            cand = list((SRC/split).glob(f"*{fname}"))
            if cand: src_img = cand[0]
        # copy image
        if src_img.exists():
            shutil.copy2(src_img, DST/"images"/split/fname)
        # write YOLO txt (one line per bbox: class x_center y_center width height — normalized)
        txt = DST/"labels"/split/(Path(fname).stem + ".txt")
        lines = []
        for ann in g.get(img_id, []):
            cname = id_to_name[ann["category_id"]]
            cid = name_to_id[cname]
            x,y,bw,bh = ann["bbox"]  # COCO xywh
            xc = (x + bw/2) / w
            yc = (y + bh/2) / h
            bw /= w; bh /= h
            # clamp 0..1
            xc, yc, bw, bh = [max(0,min(1,v)) for v in (xc,yc,bw,bh)]
            lines.append(f"{cid} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
        txt.write_text("\n".join(lines), encoding="utf-8")
    return cats

cats_train = coco_to_yolo(SRC/"train"/"_annotations.coco.json", "train")
cats_valid = coco_to_yolo(SRC/"valid"/"_annotations.coco.json", "valid")
assert cats_train == cats_valid, f"class mismatch {cats_train} vs {cats_valid}"
cats = cats_train
print(f"classes ({len(cats)}):", cats)

# write dataset.yaml (YOLO expects absolute or relative paths)
yaml = f"""path: {DST.as_posix()}
train: images/train
val: images/val
names:
"""
for i,n in enumerate(cats):
    yaml += f"  {i}: {n}\n"
(DST/"dataset.yaml").write_text(yaml, encoding="utf-8")
print(open(DST/"dataset.yaml", encoding="utf-8").read())
print(f"images train={len(list((DST/'images/train').glob('*.jpg')))} val={len(list((DST/'images/val').glob('*.jpg')))}")
print(f"labels train={len(list((DST/'labels/train').glob('*.txt')))} val={len(list((DST/'labels/val').glob('*.txt')))}")


## 3) Verify a few labels

In [ ]:
import random, cv2
from pathlib import Path
import matplotlib.pyplot as plt
p = Path("/content/dataset_yolo")
imgs = list((p/"images/train").glob("*.jpg"))
random.seed(0); samp = random.sample(imgs, min(3, len(imgs)))
for im_path in samp:
    img = cv2.cvtColor(cv2.imread(str(im_path)), cv2.COLOR_BGR2RGB)
    h,w = img.shape[:2]
    txt = p/"labels/train"/(im_path.stem + ".txt")
    for line in txt.read_text().strip().splitlines():
        if not line: continue
        cid, xc, yc, bw, bh = map(float, line.split())
        x1 = int((xc - bw/2)*w); y1 = int((yc - bh/2)*h)
        x2 = int((xc + bw/2)*w); y2 = int((yc + bh/2)*h)
        cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(img, str(int(cid)), (x1, max(0,y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 1)
    plt.figure(figsize=(6,6)); plt.imshow(img); plt.axis("off"); plt.show()


## 4) Train YOLO26 nano

T4 15 GB → `imgsz=640 batch=32` is safe. For 504-native data `imgsz=512` or `640` both work — `640` is YOLO's native.  
If `yolo26n.pt` is not yet in your ultralytics version, the code falls back to `yolo11n.pt` (same API).

In [ ]:
from ultralytics import YOLO
import pathlib

yaml_path = "/content/dataset_yolo/dataset.yaml"

# try YOLO26 nano, fallback to 11 nano
for w in ["yolo26n.pt", "yolo11n.pt", "yolov8n.pt"]:
    try:
        model = YOLO(w)
        print(f"loaded {w}")
        break
    except Exception as e:
        print(f"{w} not found: {e}")
else:
    raise FileNotFoundError("No YOLO nano weights found")

# train — adjust epochs/batch/imgsz as needed
results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,
    workers=4,
    project="/content/runs",
    name="yolo26n_green",
    exist_ok=True,
    amp=True,
    patience=20,
    optimizer="auto",
    cos_lr=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,  # augmentation tuned for green cloth + metallic reflections + shadows
    degrees=5, translate=0.05, scale=0.05, shear=1,  # keep geometry mild — shadow makes bbox hard, don't over-augment geometry
    fliplr=0.5,
    mosaic=0.5,
)
print("best:", results)


## 5) Validate

In [ ]:
from ultralytics import YOLO
model = YOLO("/content/runs/yolo26n_green/weights/best.pt")
metrics = model.val(data="/content/dataset_yolo/dataset.yaml", imgsz=640, batch=32)
print(metrics)


## 6) Predict on a few val images

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import random
model = YOLO("/content/runs/yolo26n_green/weights/best.pt")
val_imgs = list(Path("/content/dataset_yolo/images/val").glob("*.jpg"))
for p in random.sample(val_imgs, min(3, len(val_imgs))):
    r = model.predict(source=str(p), imgsz=640, conf=0.25, save=False, verbose=False)[0]
    print(p.name, [(model.names[int(c)], float(conf)) for c,conf in zip(r.boxes.cls, r.boxes.conf)] if r.boxes is not None else "no det")
    # save with boxes for visual check
    r.save(filename=f"/content/pred_{p.name}")
    from IPython.display import Image, display
    display(Image(filename=f"/content/pred_{p.name}"))


## 7) Export / Download

- `best.pt` is at `/content/runs/yolo26n_green/weights/best.pt`
- For deployment `model.export(format="onnx")`

In [ ]:
# download best.pt
from google.colab import files
files.download("/content/runs/yolo26n_green/weights/best.pt")
# or copy to Drive
# !cp "/content/runs/yolo26n_green/weights/best.pt" /content/drive/MyDrive/

# optional export
# from ultralytics import YOLO
# YOLO("/content/runs/yolo26n_green/weights/best.pt").export(format="onnx", imgsz=640)
